# Lesson 4: MCP Tools

---
## 1. Setup

In [ ]:
import httpx
import json
import socket
import time
import threading
from pathlib import Path
from typing import Optional

from fastmcp import FastMCP, Client
from pydantic import BaseModel

BASE = 'http://localhost:8000/api/v1'

def show(resp):
    """Pretty-print a response: status code + formatted JSON body."""
    print(f'HTTP {resp.status_code}')
    resp.raise_for_status()
    print(json.dumps(resp.json(), indent=2))

print('Imports ready ✓')

In [ ]:
try:
    resp = httpx.get(f'{BASE}/machines', params={'limit': 1}, timeout=3.0)
    resp.raise_for_status()
    data = resp.json()
    print(f"API server is running ✓  ({data['total_matching']} total records in index)")
except Exception as e:
    print(f'❌ API server is not responding: {e}')
    print()
    print('The Lesson 3 notebook must be running before this notebook can proceed.')
    print('Open api/lesson_03_api_design.ipynb, run all cells, and confirm the API')
    print('server started successfully (Step 3 / server polling cell).')
    print()
    print('The server runs on http://localhost:8000 — once it is up, re-run this cell.')

### The Architecture So Far

```
Lesson 2:  SQLite  →  OpenSearch index (machine_usage)
                              ↓
Lesson 3:  FastAPI  (12 endpoints)  ←  http://localhost:8000
                              ↓
Lesson 4:  MCP tools   ←  this notebook
                              ↓
Lesson 5:  AI Agent
```

The API we built in Lesson 3 is designed for programs to call. MCP tools are designed for **LLMs** to call. Same underlying data, different consumer.

---
## 2. What is MCP?

**MCP (Model Context Protocol)** is the standard protocol for giving LLMs access to tools. When an LLM wants to use a tool, it:

1. Reads the tool's **name** and **description** to decide if it's relevant
2. Reads the **parameter schema** to understand what to pass
3. Calls the tool and incorporates the result into its response

That means three things we write directly shape how well the LLM uses our tools:

| What we write | What the LLM uses it for |
|---|---|
| Tool name | Routing — which tool answers this question? |
| Tool description (docstring) | When to use it, what it returns, what NOT to use it for |
| Parameter descriptions | What values are valid, where to get them from |

`fastmcp` wraps a Python function as an MCP tool. The `@mcp.tool` decorator reads:
- The function name → tool name
- The docstring → tool description
- The type annotations and default values → parameter schema
- The `Args:` section of the docstring → parameter descriptions

**The API we built is for code to call. The MCP layer is for LLMs to call.** Same data, different consumer, different design constraints.

---
## 3. Tool Design Principles

Three rules before we write a single tool.

---

### Rule 1: Names are routing instructions

The model reads the name to decide if a tool is worth looking at.

```python
# ❌ Tells the model nothing
def maintenance_issues(...):

# ✓ Tells the model exactly when to use it
def find_top_maintenance_issues(...):
```

Use `verb_noun` or `verb_noun_by_field` form. Verbs like `rank_`, `find_`, `get_`, `compare_`, `submit_`, `analyze_` carry intent. Nouns like `_machines_by_maintenance` carry the domain.

---

### Rule 2: Descriptions are routing logic

The description is the first thing the model reads when choosing between tools. It needs to answer three questions: *What does this return? When should I call it? When should I NOT call it?*

```python
# ❌ Too generic
def rank_machines_by_maintenance(...):
    """Returns machine maintenance data."""

# ✓ Specific enough to route correctly
def rank_machines_by_maintenance(...):
    """Rank machines by total maintenance downtime hours.

    Use this as the FIRST step when investigating which machines
    have the most downtime. Follow-up: find_top_maintenance_issues.

    Do NOT use for regular operator sessions.
    """
```

---

### Rule 3: Parameters need context

The model has to decide what value to pass. A bare type annotation gives it nothing to work with.

```python
# ❌ Model doesn't know where to get this
machine_id: str | None = None

# ✓ Model knows exactly where to get it
# (In the Args: section of the docstring:)
# machine_id: Filter to a specific machine UUID.
#             Get this from rank_machines_by_maintenance.
```

Cross-references between tools close the loop: the model knows that the output of one tool becomes the input to another.

---
## 4. The 9 Lesson 3 Tools

We'll wrap every endpoint from Lesson 3 as an MCP tool, applying the three design principles to each.

First the response models (what each tool returns), then the FastMCP server instance, then the tools themselves — grouped by concern.

### Response Models

Pydantic models define the shape of each tool's return value. FastMCP uses them to serialize responses correctly, and the model names appear in the tool schema — so the LLM knows what it's receiving.

All models for the lesson are defined here in one place: the first group covers the 9 Lesson 3 tools, the second covers the 3 new tools introduced in Section 5, and the last two are for the composite tool in Section 6.

In [ ]:
from utils.models import MaintenanceDoc, MaintenancePersonnelBucket, PersonnelIssueBucket, TrainingRecommendation


class DurationStats(BaseModel):
    total_duration_hours: float
    average_duration_hours: float
    record_count: int


class MachineAggregationBucket(DurationStats):
    machine_id: str


class PersonnelAggregationBucket(DurationStats):
    personnel_id: str


class ShiftBucket(BaseModel):
    shift: str
    hours: str
    total_idle_hours: float
    maintenance_idle_hours: float
    operator_gap_idle_hours: float
    maintenance_events: int
    utilization_pct: float


class MaintenanceIssueBucket(DurationStats):
    machine_id: str
    reason_code: str


class RecordItem(BaseModel):
    id: str
    machine_id: str
    personnel_id: str
    title: str
    type: str
    start_timestamp: str
    end_timestamp: str
    duration_hours: float
    shift_name: str
    reason_code: Optional[str] = None
    employee_idle_hours_from_prev: Optional[float] = None
    machine_idle_hours_from_prev: Optional[float] = None


class TrainingAnalysisResult(BaseModel):
    recommendations: list[TrainingRecommendation]
    skipped_count: int
    issues_analyzed: int
    personnel_found: int


print('Data models defined ✓')


### Step 2: Create the FastMCP Server

`FastMCP` is the server object. Every `@mcp.tool`-decorated function registered below will be exposed as an MCP tool when the server starts.

In [ ]:
mcp = FastMCP('Machine Usage Analytics')
print('FastMCP server instance created ✓')

### Machine Discovery Tools

Three endpoints aggregate by machine — one for maintenance events, one for regular operator sessions, one for both combined. Separate tools with distinct names let the model choose the right lens without a `type` parameter it might use incorrectly.

In [ ]:
@mcp.tool
def rank_machines_by_maintenance(
    shift: str | None = None,
    limit: int = 5,
    sort_key: str = 'total_duration_hours',
    start_date: str | None = None,
    end_date: str | None = None,
    personnel_id: str | None = None,
    reason_code: str | None = None,
) -> list[MachineAggregationBucket]:
    """Rank machines by total maintenance downtime hours.

    Use this as the FIRST step when investigating which machines have the most
    downtime or are causing the most operational disruption. Returns machines
    sorted by total maintenance time (highest first by default).

    Follow-up tools: find_top_maintenance_issues (which failure codes hit this
    machine), rank_personnel_by_maintenance (who services it), get_records
    (full event timeline).

    Use reason_code to start from a known failure type instead of a known machine —
    e.g., 'which machines are most affected by bearing wear?' Get reason code UUIDs
    from find_top_maintenance_issues or get_maintenance_docs.

    Do NOT use for regular operator session analysis.

    Args:
        shift: Filter by shift ('day', 'swing', or 'night'). Omit for all shifts.
        limit: Number of machines to return (1-100, default 5).
        sort_key: Sort by 'total_duration_hours' (default), 'average_duration_hours', or 'record_count'.
        start_date: Include records on or after this date (YYYY-MM-DD).
        end_date: Include records on or before this date (YYYY-MM-DD).
        personnel_id: Filter to a specific technician UUID.
        reason_code: Filter to a specific failure reason code UUID. Get from find_top_maintenance_issues.
    """
    params: dict = {'limit': limit, 'sort_key': sort_key}
    if shift: params['shift'] = shift
    if start_date: params['start_date'] = start_date
    if end_date: params['end_date'] = end_date
    if personnel_id: params['personnel_id'] = personnel_id
    if reason_code: params['reason_code'] = reason_code
    resp = httpx.get(f'{BASE}/maintenance/machines', params=params, timeout=5.0)
    resp.raise_for_status()
    return [MachineAggregationBucket(**r) for r in resp.json()['results']]


@mcp.tool
def rank_machines_by_operator_usage(
    shift: str | None = None,
    limit: int = 5,
    sort_key: str = 'total_duration_hours',
    mode: str = 'active',
    start_date: str | None = None,
    end_date: str | None = None,
    personnel_id: str | None = None,
) -> list[MachineAggregationBucket]:
    """Rank machines by regular operator session hours (excluding maintenance).

    Use to identify which machines get the most operator attention, or filter by
    personnel_id to see which machines a specific operator uses most. Useful when
    correlating high-maintenance machines with the operators who run them.

    Use mode='idle' to rank by machine idle gaps between sessions instead of active hours.

    Do NOT use for maintenance analysis.

    Args:
        shift: Filter by shift ('day', 'swing', or 'night'). Omit for all shifts.
        limit: Number of machines to return (1-100, default 5).
        sort_key: Sort by 'total_duration_hours' (default), 'average_duration_hours', or 'record_count'.
        mode: 'active' (default) sums session duration; 'idle' sums gap hours between sessions.
        start_date: Include records on or after this date (YYYY-MM-DD).
        end_date: Include records on or before this date (YYYY-MM-DD).
        personnel_id: Filter to a specific operator UUID.
    """
    params: dict = {'limit': limit, 'sort_key': sort_key, 'mode': mode}
    if shift: params['shift'] = shift
    if start_date: params['start_date'] = start_date
    if end_date: params['end_date'] = end_date
    if personnel_id: params['personnel_id'] = personnel_id
    resp = httpx.get(f'{BASE}/regular/machines', params=params, timeout=5.0)
    resp.raise_for_status()
    return [MachineAggregationBucket(**r) for r in resp.json()['results']]


@mcp.tool
def rank_all_machines(
    shift: str | None = None,
    limit: int = 5,
    sort_key: str = 'total_duration_hours',
    mode: str = 'active',
    start_date: str | None = None,
    end_date: str | None = None,
) -> list[MachineAggregationBucket]:
    """Rank machines by total hours across all record types (Regular + Maintenance combined).

    Use for a broad overview of machine activity when you don't need to separate
    operator time from maintenance time. For focused analysis, prefer
    rank_machines_by_maintenance or rank_machines_by_operator_usage.

    Args:
        shift: Filter by shift ('day', 'swing', or 'night'). Omit for all shifts.
        limit: Number of machines to return (1-100, default 5).
        sort_key: Sort by 'total_duration_hours' (default), 'average_duration_hours', or 'record_count'.
        mode: 'active' (default) sums session duration; 'idle' sums gap hours between sessions.
        start_date: Include records on or after this date (YYYY-MM-DD).
        end_date: Include records on or before this date (YYYY-MM-DD).
    """
    params: dict = {'limit': limit, 'sort_key': sort_key, 'mode': mode}
    if shift: params['shift'] = shift
    if start_date: params['start_date'] = start_date
    if end_date: params['end_date'] = end_date
    resp = httpx.get(f'{BASE}/machines', params=params, timeout=5.0)
    resp.raise_for_status()
    return [MachineAggregationBucket(**r) for r in resp.json()['results']]


print('Machine tools registered ✓  (rank_machines_by_maintenance, rank_machines_by_operator_usage, rank_all_machines)')

### Personnel Discovery Tools

Same three-way split as machines — maintenance technicians, operators, and both combined. Notice that `rank_personnel_by_maintenance` and `rank_operators_by_usage` accept a `machine_id` filter: this is the tool chain in action — the output of a machine ranking tool becomes the input filter here.

In [ ]:
@mcp.tool
def rank_personnel_by_maintenance(
    shift: str | None = None,
    limit: int = 5,
    sort_key: str = 'total_duration_hours',
    start_date: str | None = None,
    end_date: str | None = None,
    machine_id: str | None = None,
    reason_code: str | None = None,
) -> list[PersonnelAggregationBucket]:
    """Rank maintenance technicians by total maintenance hours worked.

    Use after rank_machines_by_maintenance identifies a machine of interest.
    Filter by machine_id to see who services that specific machine most.

    Use reason_code to narrow to a specific failure type — e.g., 'who handles
    hydraulic pressure failures most?' This is less targeted than
    find_personnel_for_issues (which matches exact machine+code pairs), but
    useful for fleet-wide questions about a failure type.

    For exact (machine, failure-code) pair lookups — who handled a specific
    failure type on a specific machine — use find_personnel_for_issues instead.

    Do NOT use for operator (regular session) analysis.

    Args:
        shift: Filter by shift ('day', 'swing', or 'night'). Omit for all shifts.
        limit: Number of personnel to return (1-100, default 5).
        sort_key: Sort by 'total_duration_hours' (default), 'average_duration_hours', or 'record_count'.
        start_date: Include records on or after this date (YYYY-MM-DD).
        end_date: Include records on or before this date (YYYY-MM-DD).
        machine_id: Filter to a specific machine UUID. Get this from rank_machines_by_maintenance.
        reason_code: Filter to a specific failure reason code UUID. Get from find_top_maintenance_issues.
    """
    params: dict = {'limit': limit, 'sort_key': sort_key}
    if shift: params['shift'] = shift
    if start_date: params['start_date'] = start_date
    if end_date: params['end_date'] = end_date
    if machine_id: params['machine_id'] = machine_id
    if reason_code: params['reason_code'] = reason_code
    resp = httpx.get(f'{BASE}/maintenance/personnel', params=params, timeout=5.0)
    resp.raise_for_status()
    return [PersonnelAggregationBucket(**r) for r in resp.json()['results']]


@mcp.tool
def rank_operators_by_usage(
    shift: str | None = None,
    limit: int = 5,
    sort_key: str = 'total_duration_hours',
    mode: str = 'active',
    start_date: str | None = None,
    end_date: str | None = None,
    machine_id: str | None = None,
) -> list[PersonnelAggregationBucket]:
    """Rank operators by regular session hours.

    Use to find the operators most associated with a specific machine, or to identify
    operators with unusually high idle time (mode='idle'). Filter by machine_id to
    see which operators run a specific machine most — useful when investigating
    whether operator behavior correlates with maintenance patterns.

    Do NOT use for maintenance personnel analysis.

    Args:
        shift: Filter by shift ('day', 'swing', or 'night'). Omit for all shifts.
        limit: Number of operators to return (1-100, default 5).
        sort_key: Sort by 'total_duration_hours' (default), 'average_duration_hours', or 'record_count'.
        mode: 'active' (default) sums session duration; 'idle' sums gap hours between sessions.
        start_date: Include records on or after this date (YYYY-MM-DD).
        end_date: Include records on or before this date (YYYY-MM-DD).
        machine_id: Filter to a specific machine UUID. Get this from rank_machines_by_maintenance.
    """
    params: dict = {'limit': limit, 'sort_key': sort_key, 'mode': mode}
    if shift: params['shift'] = shift
    if start_date: params['start_date'] = start_date
    if end_date: params['end_date'] = end_date
    if machine_id: params['machine_id'] = machine_id
    resp = httpx.get(f'{BASE}/regular/personnel', params=params, timeout=5.0)
    resp.raise_for_status()
    return [PersonnelAggregationBucket(**r) for r in resp.json()['results']]


@mcp.tool
def rank_all_personnel(
    shift: str | None = None,
    limit: int = 5,
    sort_key: str = 'total_duration_hours',
    start_date: str | None = None,
    end_date: str | None = None,
    machine_id: str | None = None,
) -> list[PersonnelAggregationBucket]:
    """Rank all employees by total hours across both Regular and Maintenance records.

    Use for a broad view of personnel activity without separating operator vs
    maintenance roles. For focused analysis, prefer rank_personnel_by_maintenance
    or rank_operators_by_usage.

    Args:
        shift: Filter by shift ('day', 'swing', or 'night'). Omit for all shifts.
        limit: Number of personnel to return (1-100, default 5).
        sort_key: Sort by 'total_duration_hours' (default), 'average_duration_hours', or 'record_count'.
        start_date: Include records on or after this date (YYYY-MM-DD).
        end_date: Include records on or before this date (YYYY-MM-DD).
        machine_id: Filter to a specific machine UUID.
    """
    params: dict = {'limit': limit, 'sort_key': sort_key}
    if shift: params['shift'] = shift
    if start_date: params['start_date'] = start_date
    if end_date: params['end_date'] = end_date
    if machine_id: params['machine_id'] = machine_id
    resp = httpx.get(f'{BASE}/personnel', params=params, timeout=5.0)
    resp.raise_for_status()
    return [PersonnelAggregationBucket(**r) for r in resp.json()['results']]


print('Personnel tools registered ✓  (rank_personnel_by_maintenance, rank_operators_by_usage, rank_all_personnel)')

### Shift Comparison, Issue Analysis, and Record Retrieval

The last three Lesson 3 tools:
- `compare_shifts` gives structural context (is this a shift-specific problem or systemic?)
- `find_top_maintenance_issues` bridges discovery and diagnosis (which failure *types* are most costly?)
- `get_records` is the diagnosis tool — individual events in time order

Notice `find_top_maintenance_issues` returns `reason_code` values — UUIDs that are opaque on their own. Section 5 introduces the endpoint that unlocks what those codes mean.

In [ ]:
@mcp.tool
def compare_shifts(
    start_date: str | None = None,
    end_date: str | None = None,
) -> list[ShiftBucket]:
    """Compare utilization, maintenance downtime, and operator idle time across Day, Swing, and Night shifts.

    Use to determine whether a problem is shift-specific (a structural staffing or
    process issue) or spread evenly across all shifts (a systemic problem). A shift
    with significantly higher maintenance_idle_hours warrants separate investigation.

    Args:
        start_date: Include records on or after this date (YYYY-MM-DD).
        end_date: Include records on or before this date (YYYY-MM-DD).
    """
    params: dict = {}
    if start_date: params['start_date'] = start_date
    if end_date: params['end_date'] = end_date
    resp = httpx.get(f'{BASE}/shifts/comparison', params=params, timeout=5.0)
    resp.raise_for_status()
    return [ShiftBucket(**s) for s in resp.json()['shifts']]


@mcp.tool
def find_top_maintenance_issues(
    shift: str | None = None,
    machine_id: str | None = None,
    limit: int = 5,
    start_date: str | None = None,
    end_date: str | None = None,
) -> list[MaintenanceIssueBucket]:
    """Find top maintenance issues ranked by total downtime, grouped by machine and failure code.

    Returns (machine_id, reason_code) pairs sorted by total downtime hours. Each
    reason_code is a UUID — use get_maintenance_docs to look up what it means.

    Use after rank_machines_by_maintenance identifies a machine of interest, or
    without machine_id to find the worst issues across all machines. The results
    can be passed directly to find_personnel_for_issues.

    Do NOT assume the top result corresponds to a failure type the user named.
    This tool ranks by downtime, not by name. If the user named a failure type
    (e.g. 'coolant leak'), call find_reason_code_by_description first to resolve
    the UUID, then pass it as the reason_code filter.

    Args:
        shift: Filter by shift ('day', 'swing', or 'night'). Omit for all shifts.
        machine_id: Filter to a specific machine UUID. Get this from rank_machines_by_maintenance.
        limit: Number of issues to return (1-100, default 5).
        start_date: Include records on or after this date (YYYY-MM-DD).
        end_date: Include records on or before this date (YYYY-MM-DD).
    """
    params: dict = {'limit': limit}
    if shift: params['shift'] = shift
    if machine_id: params['machine_id'] = machine_id
    if start_date: params['start_date'] = start_date
    if end_date: params['end_date'] = end_date
    resp = httpx.get(f'{BASE}/maintenance/issues', params=params, timeout=5.0)
    resp.raise_for_status()
    return [MaintenanceIssueBucket(**r) for r in resp.json()['results']]


@mcp.tool
def get_records(
    machine_id: str | None = None,
    personnel_id: str | None = None,
    type: str | None = None,
    shift: str | None = None,
    start_date: str | None = None,
    end_date: str | None = None,
    limit: int = 20,
    sort: str = 'asc',
) -> list[RecordItem]:
    """Retrieve individual machine usage records in time order (non-aggregated).

    Use AFTER aggregation tools have identified a machine or person of interest
    and you need to see exactly what happened and when. Returns the raw event
    timeline with all fields.

    Typical workflows:
    - Machine timeline: pass machine_id to see every session (Regular + Maintenance) in order.
    - Operator history: pass personnel_id to see all sessions for one person.
    - Incident investigation: combine machine_id + type='maintenance' for repair events only.

    Do NOT use for ranking or counting — use rank_* or find_* tools for that.

    Args:
        machine_id: Filter to a specific machine UUID. Get this from rank_machines_by_maintenance.
        personnel_id: Filter to a specific employee UUID. Get this from rank_personnel_by_maintenance.
        type: Filter by record type: 'regular' (operator sessions) or 'maintenance'.
        shift: Filter by shift ('day', 'swing', or 'night'). Omit for all shifts.
        start_date: Include records on or after this date (YYYY-MM-DD).
        end_date: Include records on or before this date (YYYY-MM-DD).
        limit: Maximum records to return (1-100, default 20).
        sort: Sort direction on start_timestamp: 'asc' (default, oldest first) or 'desc'.
    """
    params: dict = {'limit': limit, 'sort': sort}
    if machine_id: params['machine_id'] = machine_id
    if personnel_id: params['personnel_id'] = personnel_id
    if type: params['type'] = type
    if shift: params['shift'] = shift
    if start_date: params['start_date'] = start_date
    if end_date: params['end_date'] = end_date
    resp = httpx.get(f'{BASE}/records', params=params, timeout=5.0)
    resp.raise_for_status()
    return [RecordItem(**r) for r in resp.json()['records']]


print('Remaining tools registered ✓  (compare_shifts, find_top_maintenance_issues, get_records)')
print('\nAll 9 Lesson 3 tools registered.')

### Try a Tool

Call one tool directly against the live API to confirm the setup is working before we continue.

In [ ]:
results = rank_machines_by_maintenance(limit=3)
print(f'rank_machines_by_maintenance returned {len(results)} buckets:')
for r in results:
    print(f'  {r.machine_id}  {r.total_duration_hours:.1f}h  ({r.record_count} events)')

---
## 5. Three New Endpoint Tools

These three tools complete the analytical pipeline. Each was deferred from Lesson 3 because it required context that didn't exist yet:

| Endpoint | What it enables |
|---|---|
| `POST /maintenance/personnel-lookup` | Targeted lookup: who handled a specific `(machine, failure-code)` pair — not just everyone who worked on the machine. |
| `POST /maintenance/docs` | Resolves opaque reason code UUIDs into descriptions, SOPs, and training materials. |
| `POST /retraining` | The only write endpoint — records a retraining recommendation, requiring explicit human approval before use. |

Each subsection explains the design rationale, then defines the tool. The API endpoints backing them were registered in Lesson 3 Section 7.

### `find_personnel_for_issues` — Targeted personnel lookup

`rank_personnel_by_maintenance` tells you everyone associated with maintenance on a machine. But who specifically was *there* when a particular failure occurred?

A note on the data model: the `personnel` field on a Maintenance record is the employee who **reported** the issue — most often the operator who was running the machine when it broke, occasionally a shift manager. There is no separate "repair technician" field in the data. So this tool answers: *which operators are repeatedly present when this specific failure type occurs on this specific machine?* That's the retraining signal.

This is the **tool chain in action**: the output of `find_top_maintenance_issues` — specifically the `machine_id` and `reason_code` fields — flows directly into the `issues` parameter here.

In [ ]:
@mcp.tool
def find_personnel_for_issues(
    issues: list[dict],
    start_date: str | None = None,
    end_date: str | None = None,
    limit: int = 10,
) -> list[MaintenancePersonnelBucket]:
    """Find employees present during specific (machine, failure-code) maintenance events.

    In the data model, the personnel field on a Maintenance record is the employee who
    *reported* the issue — typically the operator running the machine when it broke (90%),
    occasionally a shift manager (10%). This is NOT the repair technician; it is the person
    whose session was interrupted by the failure.

    Use this when you want to know: which operators are correlated with a specific failure
    type on a specific machine? If the same person appears repeatedly across a failure code,
    that is the training signal — the operator may need guidance on how to prevent or
    recognize that failure type earlier.

    More targeted than rank_personnel_by_maintenance: matches on exact (machine_id,
    reason_code) pairs and returns a per-issue breakdown for each person found.

    Use after find_top_maintenance_issues identifies the costly failure patterns.
    Pass those results directly as the issues parameter.

    Args:
        issues: List of issue pairs to look up. Each dict must have 'machine_id' and
                'reason_code' keys. Get these from find_top_maintenance_issues results.
                Example: [{"machine_id": "uuid...", "reason_code": "uuid..."}]
        start_date: Include records on or after this date (YYYY-MM-DD).
        end_date: Include records on or before this date (YYYY-MM-DD).
        limit: Maximum personnel to return (1-100, default 10).
    """
    payload: dict = {'issues': issues, 'limit': limit}
    if start_date: payload['start_date'] = start_date
    if end_date: payload['end_date'] = end_date
    resp = httpx.post(f'{BASE}/maintenance/personnel-lookup', json=payload, timeout=5.0)
    resp.raise_for_status()
    return [MaintenancePersonnelBucket(**r) for r in resp.json()['results']]


print('find_personnel_for_issues registered ✓')

### `get_maintenance_docs` — Unlock the reason codes

In Lesson 3, we noticed that every maintenance event has a `reason_code` — but it's just a UUID. Opaque. We knew *that* a failure occurred, not *what* it was.

This endpoint resolves that. Each reason code maps to up to four document types: a technical description, a standard operating procedure, training materials for operators, and vendor support contacts. It reads from reference data, not OpenSearch — which is why it wasn't in Lesson 3.

In [ ]:
@mcp.tool
def get_maintenance_docs(
    reason_codes: list[str],
    limit: int = 5,
) -> list[MaintenanceDoc]:
    """Retrieve documentation for maintenance failure codes (descriptions, SOPs, training materials, support contacts).

    Each reason_code UUID maps to up to 4 document types:
    - 'description': technical summary of what this failure is
    - 'SOP': standard operating procedure for handling it
    - 'training': training material for operators
    - 'support': vendor or specialist contact information

    Use after find_top_maintenance_issues or find_personnel_for_issues surface
    reason codes of interest, to understand WHAT the failure is and find
    actionable remediation resources.

    Args:
        reason_codes: List of reason code UUIDs to look up. Get these from
                      find_top_maintenance_issues or find_personnel_for_issues output.
        limit: Maximum documents per reason code (1-20, default 5).
    """
    resp = httpx.post(
        f'{BASE}/maintenance/docs',
        json={'reason_codes': reason_codes, 'limit': limit},
        timeout=5.0,
    )
    resp.raise_for_status()
    return [MaintenanceDoc(**d) for d in resp.json()['docs']]


print('get_maintenance_docs registered ✓')

### `find_reason_code_by_description` — Failure name → UUID

`get_maintenance_docs` resolves a UUID into a description. The reverse is just as common: the user names a failure type and the agent needs the matching `reason_code` to filter any other tool.

Without this tool, the agent has to fetch every doc from `find_top_maintenance_issues` results and scan descriptions itself. In practice it takes shortcuts — anchoring on the top-ranked code regardless of whether it matches the user's query — and confidently reports the wrong failure type.

This tool reads directly from the reference dictionary in `utils/reference_data.py` (no API call needed). Not every MCP tool has to wrap an HTTP endpoint.

In [ ]:
from utils.reference_data import CANNED_DOCS

@mcp.tool
def find_reason_code_by_description(query: str) -> list[MaintenanceDoc]:
    """Resolve a failure-type name (e.g. 'coolant leak', 'bearing wear') to its reason_code UUID.

    Use this FIRST whenever the user names a failure type in natural language,
    BEFORE calling any tool that takes a reason_code parameter
    (rank_machines_by_maintenance, rank_personnel_by_maintenance,
    find_top_maintenance_issues with reason_code, get_records).

    Do NOT assume the top result of find_top_maintenance_issues matches the
    failure type the user named — that tool ranks by downtime, not by name.

    Performs a case-insensitive substring match against the description
    document of each reason code.

    Args:
        query: Failure type to search for (e.g. 'coolant', 'hydraulic pressure').
               Matched as a substring against each reason code's description.
    """
    q = query.lower()
    matches: list[MaintenanceDoc] = []
    for items in CANNED_DOCS.values():
        for item in items:
            if item.reason_type == 'description' and q in item.document.lower():
                matches.append(item)
                break
    return matches


print('find_reason_code_by_description registered ✓')


### `submit_retraining_request` — The action endpoint

Until now, every tool has been read-only. This one writes.

The description here is intentionally heavy on constraints. When an LLM has a write tool available, it needs to understand exactly when it is and isn't appropriate to use it. The description is the only mechanism we have to set that boundary — it needs to be explicit.

Notice the tool description doesn't just say "requires approval" — it spells out *what must be true* before calling it. That specificity is what prevents a well-intentioned model from submitting retraining requests based on thin evidence.

In [ ]:
@mcp.tool
def submit_retraining_request(
    machine_id: str,
    personnel_id: str,
    reason: str,
    recommendation_basis: str | None = None,
) -> dict:
    """⚠️  HIGH-IMPACT WRITE OPERATION. Submit a retraining requirement for a specific person on a specific machine.

    This writes permanently to the database. Only call when ALL of the following are true:
    1. Analysis (via analyze_training_needs) has confirmed a specific person shows a
       pattern of above-average maintenance durations or recurring failures.
    2. The human user has reviewed the evidence and explicitly confirmed the submission.
    3. You have a specific machine_id and personnel_id from the data, not general observations.

    Do NOT call for exploratory questions, general concerns, or without explicit
    human confirmation. Always run analyze_training_needs first and present the
    approval_prompt to the user before calling this tool.

    Args:
        machine_id: UUID of the machine associated with the training gap.
        personnel_id: UUID of the employee requiring retraining.
        reason: Clear explanation of why retraining is needed (10-500 characters).
        recommendation_basis: Optional context from the analysis — specific events,
                              durations, or patterns that support this recommendation.
    """
    resp = httpx.post(
        f'{BASE}/retraining',
        json={
            'machine_id': machine_id,
            'personnel_id': personnel_id,
            'reason': reason,
            'recommendation_basis': recommendation_basis,
        },
        timeout=5.0,
    )
    resp.raise_for_status()
    return resp.json()


print('submit_retraining_request registered ✓')

### The Tool Chain in Action

These three tools are designed to work in sequence. Here's the full chain: find the top machine, get its worst failure, see who handled it, then look up what the failure actually means.

**Hypothetical query:** *"Which machine has the most maintenance downtime, what failure type is driving it, and who keeps showing up when it breaks?"*

The agent receiving this query has no hardcoded logic — it reads the tool descriptions and infers the right sequence:

```
rank_machines_by_maintenance(limit=1)
    ↓  returns the machine_id with the most downtime
find_top_maintenance_issues(machine_id=<uuid>)
    ↓  returns the (machine_id, reason_code) pairs driving that downtime
find_personnel_for_issues(issues=[{machine_id, reason_code}])
    ↓  returns personnel correlated with those specific failures
```

Each tool's output becomes the next tool's input — `machine_id` flows from ranking into issues; the `(machine_id, reason_code)` pair flows from issues into personnel lookup. This works because the `Args:` sections in each tool description explicitly tell the model where to get those values from.


In [ ]:
# Step 1: find the top machine
top_machines = rank_machines_by_maintenance(limit=1)
top_machine_id = top_machines[0].machine_id
print(f'Top machine: {top_machine_id}')

# Step 2: find its top failure issue
issues = find_top_maintenance_issues(machine_id=top_machine_id, limit=1)
top_reason_code = issues[0].reason_code
print(f'Top reason code: {top_reason_code}  ({issues[0].total_duration_hours:.1f}h total)')

# Step 3: who handled that specific issue?
issue_pairs = [{'machine_id': top_machine_id, 'reason_code': top_reason_code}]
personnel = find_personnel_for_issues(issues=issue_pairs, limit=3)
print(f'\nPersonnel who handled this issue ({len(personnel)} found):')
for p in personnel:
    print(f'  {p.personnel_id}  {p.total_duration_hours:.1f}h  ({p.record_count} events)')

# Step 4: what does the reason code actually mean?
docs = get_maintenance_docs(reason_codes=[top_reason_code], limit=2)
print(f'\nDocumentation for {top_reason_code}:')
for d in docs:
    print(f'  [{d.reason_type}] {d.document[:100]}...')

---
## 6. The Composite Tool: `analyze_training_needs`

Some workflows are predictable enough that you can encode them as a single tool call instead of making the agent figure out the sequence every time. `analyze_training_needs` runs the full analysis pipeline — issues → personnel → documentation → thresholds — in one call and returns structured recommendations ready for human review.

### Why a composite tool?

The pipeline has a specific structure with business-rule logic at the end:

```
find_top_maintenance_issues
        ↓
find_personnel_for_issues  (passing all issue pairs at once)
        ↓
get_maintenance_docs  (for all unique reason codes)
        ↓
apply thresholds  (avg_hours > 3.0 → recommend, > 4.0 → strongly recommend)
        ↓
build approval_prompt  (formatted evidence for human reviewer)
```

The threshold logic lives in `utils/training_recommender.py` — deterministic business rules that shouldn't be left to the LLM to interpret. A composite tool runs them reliably every time.

In Lesson 5 we'll see how an agent chains individual tools itself. For complex workflows with fixed business logic like this one, a composite tool is more reliable.

In [ ]:
from utils.training_recommender import build_recommendations

print('training_recommender loaded ✓')

In [ ]:
@mcp.tool
def analyze_training_needs(
    start_date: str | None = None,
    end_date: str | None = None,
    top_issues_limit: int = 5,
) -> TrainingAnalysisResult:
    """Analyze maintenance data and produce ranked training recommendations.

    Runs the complete analysis pipeline in a single call:
    1. Finds the top maintenance issues by total downtime
    2. Identifies which personnel handled those specific issues
    3. Fetches documentation for each failure code
    4. Applies business-rule thresholds to produce recommendations
       (avg duration > 3h per event → recommend; > 4h → strongly recommend)

    Each recommendation includes an approval_prompt with all the context a human
    reviewer needs. ALWAYS present the approval_prompt to the user and wait for
    explicit confirmation before calling submit_retraining_request.

    This tool is READ-ONLY. Use it before any write operations.

    Args:
        start_date: Include records on or after this date (YYYY-MM-DD).
        end_date: Include records on or before this date (YYYY-MM-DD).
        top_issues_limit: How many top maintenance issues to analyze (default 5).
    """
    issues_params: dict = {'limit': top_issues_limit}
    if start_date: issues_params['start_date'] = start_date
    if end_date: issues_params['end_date'] = end_date

    resp = httpx.get(f'{BASE}/maintenance/issues', params=issues_params, timeout=5.0)
    resp.raise_for_status()
    issues = resp.json().get('results', [])
    if not issues:
        return TrainingAnalysisResult(
            recommendations=[], skipped_count=0, issues_analyzed=0, personnel_found=0
        )

    issue_pairs = [{'machine_id': i['machine_id'], 'reason_code': i['reason_code']} for i in issues]
    payload: dict = {'issues': issue_pairs, 'limit': 50}
    if start_date: payload['start_date'] = start_date
    if end_date: payload['end_date'] = end_date

    resp = httpx.post(f'{BASE}/maintenance/personnel-lookup', json=payload, timeout=5.0)
    resp.raise_for_status()
    personnel_results = [
        MaintenancePersonnelBucket(**r) for r in resp.json().get('results', [])
    ]
    if not personnel_results:
        return TrainingAnalysisResult(
            recommendations=[], skipped_count=0,
            issues_analyzed=len(issues), personnel_found=0
        )

    reason_codes = list({i['reason_code'] for i in issues})
    try:
        resp = httpx.post(
            f'{BASE}/maintenance/docs',
            json={'reason_codes': reason_codes},
            timeout=5.0,
        )
        resp.raise_for_status()
        docs_data = [MaintenanceDoc(**d) for d in resp.json().get('docs', [])]
    except Exception:
        docs_data = []

    recs = build_recommendations(personnel_results, docs_data)
    total_pairs = sum(len(p.issues) for p in personnel_results)

    return TrainingAnalysisResult(
        recommendations=recs,
        skipped_count=total_pairs - len(recs),
        issues_analyzed=len(issues),
        personnel_found=len(personnel_results),
    )


print('analyze_training_needs registered ✓')
print('\nAll 14 tools registered.')


### Run the Full Analysis

Call `analyze_training_needs` to see the complete pipeline in action. Each recommendation includes an `approval_prompt` — the pre-built evidence block shown to the human reviewer before any retraining request is submitted.

In [ ]:
analysis = analyze_training_needs(top_issues_limit=3)

print(f'Issues analyzed:   {analysis.issues_analyzed}')
print(f'Personnel found:   {analysis.personnel_found}')
print(f'Recommendations:   {len(analysis.recommendations)}')
print(f'Skipped (low sig): {analysis.skipped_count}')

if analysis.recommendations:
    print('\n--- First approval prompt ---')
    print(analysis.recommendations[0].approval_prompt)

---
## 7. Start the MCP Server

First we'll inspect all registered tools using fastmcp's in-process client — no server required yet. Then we start the HTTP server in a background thread and confirm it's accepting connections.

In [ ]:
async def show_tools():
    """List all registered tools with full descriptions and parameter schemas."""
    async with Client(mcp) as client:
        tools = await client.list_tools()

    print(f'Registered MCP tools: {len(tools)}')
    for tool in sorted(tools, key=lambda t: t.name):
        print('─' * 62)
        print(f'TOOL: {tool.name}')
        print(f'\nDescription:')
        print((tool.description or '').strip())
        print(f'\nParameter schema:')
        print(json.dumps(tool.inputSchema, indent=2))
        print()

await show_tools()


The output above is exactly what the LLM receives before its first tool call — the full **description** and **parameter schema** for every registered tool.

- The **description** is routing logic. The model reads it to answer: "is this the right tool for the question?" Vague descriptions mean wrong choices. A description that says *when NOT to use this tool* is as important as saying when to use it.
- The **parameter schema** tells the model how to format the call. Parameter descriptions like `Get this from find_top_maintenance_issues` are what make chains work — they tell the model where to find the value, not just what type it should be.

Reading through these as a class is one of the best exercises in Section 3 tool design: if *you* can't tell from the description when to call a tool, the model won't be able to either.

Now start the HTTP server. It runs on port **8001** (port 8000 is already taken by the API server).


In [ ]:
def _run_mcp_server():
    import asyncio
    asyncio.set_event_loop(asyncio.new_event_loop())
    mcp.run(transport='sse', host='0.0.0.0', port=8001)

# Only start if port 8001 is not already in use (safe to re-run cell)
_check = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
_already_running = _check.connect_ex(('localhost', 8001)) == 0
_check.close()

if _already_running:
    print('MCP server already running on port 8001 ✓')
else:
    _mcp_thread = threading.Thread(target=_run_mcp_server, daemon=True)
    _mcp_thread.start()
    print('MCP server starting on port 8001...')

In [ ]:
print('Waiting for MCP server...', end='', flush=True)
for attempt in range(20):
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(1.0)
        result = sock.connect_ex(('localhost', 8001))
        sock.close()
        if result == 0:
            print(f' up! (attempt {attempt + 1})')
            print('MCP server ready ✓  →  http://localhost:8001')
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(1)
else:
    print('\nMCP server did not respond after 20s.')
    print('Check the output above for errors.')

---
## 8. Agent Demo: Tool Call Chain

Now we'll see what happens when an LLM drives the tools instead of us calling them directly.

The `openai-agents` SDK creates an agent backed by our MCP server. The agent receives a single query, decides which tools to call and in what order, and produces a final answer.

This is a preview of Lesson 5. The query below starts from a known failure type (hydraulic pressure loss) rather than a machine — demonstrating the `reason_code` filter path we just added to the maintenance endpoints. This requires at least 4 sequential tool calls:

```
get_maintenance_docs  →  rank_machines_by_maintenance(reason_code)  →  find_top_maintenance_issues  →  find_personnel_for_issues
```

The agent first resolves the failure type name to a UUID via docs, then uses that UUID as a filter. Watch the tool calls to see how the model chains them.

> **Prerequisite:** Set `OPENAI_API_KEY` in a `.env` file at the repo root before running this section.

In [ ]:
import os
from dotenv import load_dotenv

# Load from .env at repo root
_env_path = Path().resolve().parent / '.env'
if not _env_path.exists():
    _env_path = Path().resolve() / '.env'
load_dotenv(_env_path)

if os.environ.get('OPENAI_API_KEY'):
    print('OPENAI_API_KEY loaded ✓')
else:
    print('⚠️  OPENAI_API_KEY not found.')
    print(f'   Create a .env file at {_env_path.parent} with: OPENAI_API_KEY=sk-...')

In [ ]:
from agents import Agent, Runner
from agents.mcp import MCPServerSse, MCPServerSseParams

DEMO_QUERY = (
    'Hydraulic pressure loss is a known problem on our floor. '
    'Which machines are most affected by it, and who are the technicians handling those repairs? '
)

async def run_demo(query: str):
    async with MCPServerSse(
        params=MCPServerSseParams(url='http://localhost:8001/sse')
    ) as mcp_server:
        agent = Agent(
            name='maintenance_analyst',
            instructions=(
                'You are a manufacturing maintenance analyst. '
                'Use the available tools to investigate machine usage and maintenance data. '
                'When the user names a failure type (e.g. "coolant leak", "bearing wear"), '
                'always resolve it to a reason_code via find_reason_code_by_description '
                'BEFORE filtering or ranking — never assume the top-ranked issue matches '
                'the named type. '
                'Start broad (rank/compare), then drill into specifics. '
                'Cite specific IDs and metrics from the data.'
            ),
            mcp_servers=[mcp_server],
        )

        result = Runner.run_streamed(agent, query)

        async for event in result.stream_events():
            if type(event).__name__ == 'RawResponsesStreamEvent':
                continue

            if not hasattr(event, 'item'):
                continue

            item = event.item
            item_type = type(item).__name__

            if item_type == 'ToolCallItem':
                raw = item.raw_item
                name = raw.get('name') if isinstance(raw, dict) else getattr(raw, 'name', '?')
                args_raw = raw.get('arguments', '{}') if isinstance(raw, dict) else getattr(raw, 'arguments', '{}')
                try:
                    args = json.loads(args_raw) if isinstance(args_raw, str) else args_raw
                    args_display = json.dumps(args, indent=5)
                except Exception:
                    args_display = str(args_raw)
                print(f'\n🔧 {name}')
                print(f'   input  → {args_display}')

            elif item_type == 'ToolCallOutputItem':
                out = item.output
                try:
                    if isinstance(out, list):
                        serialized = [r.model_dump() if hasattr(r, 'model_dump') else r for r in out]
                        out_str = json.dumps(serialized, indent=5)
                    elif hasattr(out, 'model_dump'):
                        out_str = json.dumps(out.model_dump(), indent=5)
                    else:
                        out_str = str(out)
                except Exception:
                    out_str = str(out)
                print(f'   output → {out_str}')

        return result


print(f'Query: "{DEMO_QUERY}"')
print('═' * 62)
demo_result = await run_demo(DEMO_QUERY)

In [ ]:
print('═' * 62)
print('Final answer:\n')
print(demo_result.final_output)

### What just happened

The agent read our tool descriptions, decided which tools to call and in what order, passed outputs from one tool as inputs to the next, and produced a coherent answer — all without us prescribing the sequence.

The actual chain it ran:

```
find_top_maintenance_issues(limit=20)
    ↓  broad scan — surfaces all top (machine, failure) pairs by downtime
get_maintenance_docs(reason_codes=["af11c1e7..."])
    ↓  confirms what that UUID actually means: hydraulic pressure loss
find_personnel_for_issues(issues=[{machine_id, reason_code}, ...])
    ↓  finds operators correlated with that failure across the top two affected machines
```

Notice the agent didn't start with `get_maintenance_docs` even though the query named the failure type. It started with a data scan instead — found hydraulic pressure by its impact rank, extracted the UUID from the results, and only then looked up the docs to confirm what it was looking at. That's a reasonable approach: it found the failure through data, not by trusting that the human-readable name would resolve correctly.

That's exactly the value of the tool design work in Section 3:
- The tool **names** told the model where to start
- The **descriptions** told it when to follow up
- The cross-references in the **`Args:`** sections told it how to pass results forward

In **Lesson 5**, we'll build the full production agent: an agentic architecture where a `factory-analyst` agent hands off to a specialized `training_recommender` agent for the analysis-to-approval workflow, with human-in-the-loop approval built into the SDK itself — so `submit_retraining_request` pauses and prompts the user before writing anything.


---
## 9. Summary

| Section | What we did |
|---------|-------------|
| What is MCP | Understood the protocol layer between LLM and API; why tool definitions shape model behavior |
| Tool design | Three rules: names route, descriptions gate, parameters cross-reference |
| 9 Lesson 3 tools | Wrapped every endpoint with agent-friendly names, descriptions, and Args sections |
| 3 new endpoint tools | Wrapped `personnel-lookup`, `docs`, and `retraining` as MCP tools |
| Composite tool | `analyze_training_needs` orchestrates the full pipeline with deterministic business-rule thresholds |
| MCP server | Started on port 8001; reviewed tool descriptions as the LLM will see them |
| Agent demo | Watched a live tool call chain — the LLM routed itself through 3 tools to answer one question |

**Lesson 5:** Build the production agent — an agentic architecture with a factory analyst, a training recommender, and human-in-the-loop approval for every retraining submission.

In [ ]:
while True:
    query = input('Query (or "quit"): ').strip()
    if query.lower() in {'quit', 'exit', 'q'}:
        print('Exiting loop.')
        break
    if not query:
        continue
    print(f'Query: "{query}"')
    print('═' * 62)
    try:
        print('=-' * 32)
        print('Running agent...')
        demo_result = await run_demo(query)
        print('═' * 62)
        print('Final answer:\n')
        print(demo_result.final_output)
    except Exception as e:
        print(f'❌ Error: {e}')
    print()
